In [1]:
from pyspark.sql import functions as F
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("silver.address.dq")

# ── Read Bronze ───────────────────────────────────────────────────────
df_address = spark.read.table("lh_adventureworks_bronze.dbo.Person_Address")

logger.info(f"Address rows:{df_address.count()}")
logger.info(f"{df_address.printSchema()}")

try:
    df_stateprovince = spark.read.table("lh_adventureworks_bronze.dbo.Person_StateProvince")
    has_stateprovince = True
    logger.info(f"StateProvince rows: {df_stateprovince.count()}")
    logger.info(f"StateProvince rows: {df_stateprovince.count()}")
except Exception:
    has_stateprovince = False
    logger.info("Note: Person_StateProvince not in Bronze")

StatementMeta(, bcf5928f-0be3-4694-9943-eb3145e8b276, 3, Finished, Available, Finished, False)

root
 |-- AddressID: integer (nullable = true)
 |-- AddressLine1: string (nullable = true)
 |-- AddressLine2: string (nullable = true)
 |-- City: string (nullable = true)
 |-- StateProvinceID: integer (nullable = true)
 |-- PostalCode: string (nullable = true)
 |-- SpatialLocation: string (nullable = true)
 |-- rowguid: string (nullable = true)
 |-- ModifiedDate: timestamp (nullable = true)



INFO:silver.address.dq:StateProvince rows: 181
INFO:silver.address.dq:StateProvince rows: 181


In [5]:
from pyspark.sql import functions as F

# ── Join Address + StateProvince ──────────────────────────────────────
df_sp_clean = df_stateprovince.select(
    F.col("StateProvinceID").alias("sp_StateProvinceID"),
    F.col("StateProvinceCode"),
    F.col("Name").alias("StateProvinceName"),
    F.col("CountryRegionCode"),
    F.col("TerritoryID").alias("sp_TerritoryID")
)

df_silver_address = df_address.join(
    df_sp_clean,
    df_address["StateProvinceID"] == df_sp_clean["sp_StateProvinceID"],
    how="left"
).drop("sp_StateProvinceID")

# ── Select final columns ──────────────────────────────────────────────
df_silver_address = df_silver_address.select(
    "AddressID",
    "AddressLine1",
    "AddressLine2",
    "City",
    "StateProvinceID",
    "StateProvinceCode",
    "StateProvinceName",
    "CountryRegionCode",
    F.col("sp_TerritoryID").alias("TerritoryID"),
    "PostalCode",
    "SpatialLocation",
    "rowguid",
    "ModifiedDate"
)

actual_rows = df_silver_address.count()
logger.info(f"Row count: {actual_rows}")
logger.info(f"{df_silver_address.printSchema()}")

StatementMeta(, bcf5928f-0be3-4694-9943-eb3145e8b276, 7, Finished, Available, Finished, False)

root
 |-- AddressID: integer (nullable = true)
 |-- AddressLine1: string (nullable = true)
 |-- AddressLine2: string (nullable = true)
 |-- City: string (nullable = true)
 |-- StateProvinceID: integer (nullable = true)
 |-- StateProvinceCode: string (nullable = true)
 |-- StateProvinceName: string (nullable = true)
 |-- CountryRegionCode: string (nullable = true)
 |-- TerritoryID: integer (nullable = true)
 |-- PostalCode: string (nullable = true)
 |-- SpatialLocation: string (nullable = true)
 |-- rowguid: string (nullable = true)
 |-- ModifiedDate: timestamp (nullable = true)



In [6]:
# ── DQ checks ─────────────────────────────────────────────────────────
dq_passed = True
dq_results = []

def log_check(name, passed, message):
    global dq_passed
    status = "PASS" if passed else "FAIL"
    log_msg = f"[DQ] [{status}] {name} — {message}"
    if passed:
        logger.info(log_msg)
    else:
        logger.error(log_msg)
        dq_passed = False
    dq_results.append({"check": name, "status": status, "message": message})

log_check(
    "Row count",
    actual_rows == df_address.count(),
    f"Expected {df_address.count():,}, got {actual_rows:,}"
)

log_check(
    "Null PK — AddressID",
    df_silver_address.filter(F.col("AddressID").isNull()).count() == 0,
    f"{df_silver_address.filter(F.col('AddressID').isNull()).count():,} nulls"
)

dup_pk = actual_rows - df_silver_address.select("AddressID").distinct().count()
log_check("Duplicate PK — AddressID", dup_pk == 0, f"{dup_pk:,} duplicates")

unmatched_sp = df_silver_address.filter(
    F.col("StateProvinceID").isNotNull() & F.col("StateProvinceName").isNull()
).count()
log_check(
    "StateProvince join coverage",
    unmatched_sp == 0,
    f"{unmatched_sp:,} addresses with StateProvinceID but no StateProvinceName"
)

failed_checks = [r for r in dq_results if r["status"] == "FAIL"]
if failed_checks:
    failed_names = ", ".join([r["check"] for r in failed_checks])
    raise Exception(
        f"silver.address DQ failed — {len(failed_checks)} check(s) failed: "
        f"{failed_names}. Pipeline halted."
    )

logger.info(f"[DQ] All checks passed — {actual_rows:,} rows cleared for Silver write.")

StatementMeta(, bcf5928f-0be3-4694-9943-eb3145e8b276, 8, Finished, Available, Finished, False)

INFO:silver.address.dq:[DQ] [PASS] Row count — Expected 19,614, got 19,614
INFO:silver.address.dq:[DQ] [PASS] Null PK — AddressID — 0 nulls
INFO:silver.address.dq:[DQ] [PASS] Duplicate PK — AddressID — 0 duplicates
INFO:silver.address.dq:[DQ] [PASS] StateProvince join coverage — 0 addresses with StateProvinceID but no StateProvinceName
INFO:silver.address.dq:[DQ] All checks passed — 19,614 rows cleared for Silver write.


In [8]:
# ── Write ──────────────────────────────────────────────────────────────
spark.sql("CREATE SCHEMA IF NOT EXISTS lh_adventureworks_silver.person")

df_silver_address.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("lh_adventureworks_silver.person.address")

df_verify = spark.read.table("lh_adventureworks_silver.person.address")
logger.info(f"silver.person.address written — {df_verify.count():,} rows verified.")

StatementMeta(, bcf5928f-0be3-4694-9943-eb3145e8b276, 10, Finished, Available, Finished, False)

INFO:silver.address.dq:silver.person.address written — 19,614 rows verified.
